# 19 — Fair value gaps

A **fair value gap** is a band of price one side skipped. Three candles: if the third's **low** is
still above the first's **high**, nothing traded in between — the middle candle displaced through
it. That band is a *bullish* gap. Mirror it for a *bearish* one.

```
bullish:  low[i]  > high[i-2]      band = (high[i-2], low[i])     sits BELOW price
bearish:  high[i] < low[i-2]       band = (high[i],  low[i-2])    sits ABOVE price
```

The gap is knowable at the **close of the third candle** and is drawn from there — never from the
middle candle where it visually sits, which would be a two-bar lookahead painted on the chart.

This notebook does two things: it **draws** gaps from several timeframes on one chart, and it runs
notebook 18's trendline sim with an added **FVG veto**, then shows you exactly which trades the veto
removed and what they would have done.

> Nothing here is evidence of an edge. Every other pattern family in this project has been measured
> and killed — see [`docs/WHAT_WORKS.md`](../docs/WHAT_WORKS.md). The gap detector is a standard
> definition and the gates are the usual practitioner filter; whether any of it predicts anything on
> this data is what the veto comparison at the bottom is for.

## Config

Every setting lives in [`nb19_config.py`](nb19_config.py) next to this notebook — **edit it there**.

It is a plain file, not a notebook cell, because this notebook is regenerated from a script and a
script that states values re-imposes them on every rebuild. Nothing regenerates that file.

Edited it while the kernel was running? Re-run this cell — it reloads.

In [1]:
import sys, os, importlib, warnings; warnings.filterwarnings('ignore')
try:
    import quant
except ModuleNotFoundError:
    sys.path.insert(0, os.path.abspath('..')); import quant
import numpy as np, pandas as pd

# A kernel that imported `quant` before a feature existed keeps the OLD module in memory while
# Jupyter shows you the NEW file, so the traceback points at code that is not running.
import quant.indicators.fvg as _f
assert hasattr(_f, "add_fvgs"), "Kernel -> Restart: quant was imported before fair value gaps"

for _p in (os.getcwd(), os.path.join(os.getcwd(), "notebooks"), os.path.abspath("..")):
    if os.path.isfile(os.path.join(_p, "nb19_config.py")) and _p not in sys.path:
        sys.path.insert(0, _p)
import nb19_config as CFG
importlib.reload(CFG)                       # so an edit lands without restarting the kernel
globals().update({k: v for k, v in vars(CFG).items() if k.isupper()})

# A setting APPENDED to the config is invisible to an editor tab you already had open: saving that
# stale buffer writes the old tail back and the new line is gone. So a setting added later also has
# a DEFAULT here — an older config file still runs, and the notebook says which line to paste in
# rather than dying on a NameError two cells later.
SETTING_DEFAULTS = {}
_missing = {k: v for k, v in SETTING_DEFAULTS.items() if not hasattr(CFG, k)}
globals().update(_missing)
if _missing:
    print("not in your config, using the default -- paste into nb19_config.py to change it:")
    for k, v in _missing.items():
        print(f"    {k} = {v!r}")

print(f"{SYMBOL} {TF}  {VIEW_START} -> {END}")
print(f"gaps from {TF}+{'+'.join(FVG_TFS)}   mitigation={FVG_MITIGATION}   "
      f"min size={FVG_MIN_SIZE_ATR} ATR")
print(f"veto: {'ON  ' + FVG_VETO_MODE if USE_FVG_VETO else 'OFF'}")
print(f"settings from {CFG.__file__}")

BTCUSDT 5m  2026-06-01 -> 2026-06-10
gaps from 5m+15m+1h+4h   mitigation=ce   min size=0.5 ATR
veto: ON  retrace
settings from C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks\nb19_config.py


## Run

In [2]:
from quant.data import get_ohlcv, tf_frames
from quant.indicators import add_mtf_emas, fvg_table, mtf_fvgs
from quant.indicators.trendlines import mtf_trendlines
from quant.scenarios import run_pieces
from quant.scenarios.entries import trendline_trigger
from quant.strategies import fvg_clear
from quant.strategies.conditions import raw
from quant.strategies.rulespec import build_rules, describe_rules, required_emas

df = get_ohlcv(SYMBOL, TF, start=WARMUP_START, end=END, source=SOURCE, tz="UTC").reset_index(drop=True)

# LINE frames are what trendlines are fitted on -- and therefore what gets TRADED.
LINE_FRAMES = tf_frames(SYMBOL, LINE_TFS, view_start=VIEW_START, end=END,
                        warmup_bars=TF_WARMUP_BARS, source=SOURCE) if LINE_TFS else {}
# GAP frames are drawn and may veto, but are never fitted. Kept separate on purpose: adding 4h to
# LINE_TFS just to see its gaps would quietly start trading 4h trendlines.
GAP_FRAMES = tf_frames(SYMBOL, tuple(FVG_TFS), view_start=VIEW_START, end=END,
                       warmup_bars=TF_WARMUP_BARS, source=SOURCE) if FVG_TFS else {}

FVG_KW = dict(mitigation=FVG_MITIGATION, min_size_atr=FVG_MIN_SIZE_ATR,
              min_disp_atr=FVG_MIN_DISP_ATR, max_age_bars=FVG_MAX_AGE_BARS,
              atr_period=FVG_ATR_PERIOD)

D = mtf_trendlines(df, LINE_FRAMES, **TRENDLINE)
D = add_mtf_emas(D, {**LINE_FRAMES, **GAP_FRAMES}, required_emas(ENTRY_RULES, chart_tf=TF))
# the gap COLUMNS, on the chart's grid, with the usual +1 higher-timeframe shift
_fv = mtf_fvgs(df, GAP_FRAMES, **FVG_KW)
for c in [c for c in _fv.columns if "fvg_" in c]:
    D[c] = _fv[c].to_numpy()

TF_NAMES = {"": TF, **{f"{k}__": k for k in LINE_FRAMES}}
PARAMS = dict(tl_prefixes=tuple(TF_NAMES), tl_tf_names=TF_NAMES,
              tl_close_n=TL_CLOSE_N, tl_full_m=TL_FULL_M, tl_touch_k=TL_TOUCH_K,
              tl_dir_n=TL_DIR_N, tl_dir_src=TL_DIR_SRC,
              tl_dir_full_n=TL_DIR_FULL_N, tl_extend=TL_EXTEND,
              tl_combine=TL_COMBINE, tl_tol_atr=TL_TOL_ATR,
              stop_n=STOP_N, stop_buffer_pct=STOP_BUFFER_PCT, be_at_rr=BE_AT_RR, tp_rr=TP_RR,
              sec_touch_n=SEC_TOUCH_N, sec_wrong_m=SEC_WRONG_M, sec_opp_n=SEC_OPP_N,
              sec_min_pct=SEC_MIN_PCT, sec_replace_tp=SEC_REPLACE_TP)
SECONDARY = "entry_retest_bail@v1" if USE_RETEST_BAIL else None

_vs = pd.Timestamp(VIEW_START, tz="UTC")
IN_WINDOW = raw(lambda d: (pd.to_datetime(d["t"]) >= _vs).to_numpy(), "in_window")

def _filter(side, veto):
    # None means "no rule for this side", which is NOT the same as "nothing passes"
    extra = build_rules(ENTRY_RULES, side, chart_tf=TF, combine=RULES_COMBINE)
    out = IN_WINDOW if extra is None else (IN_WINDOW & extra)
    if veto:
        out = out & fvg_clear(side, mode=FVG_VETO_MODE, timeframes=FVG_VETO_TFS,
                              max_age_bars=FVG_VETO_MAX_AGE, min_size_atr=FVG_VETO_MIN_SIZE,
                              max_dist_atr=FVG_VETO_MAX_DIST)
    return out

def _run(veto):
    sides = [s for s, on in (("long", TRADE_LONG), ("short", TRADE_SHORT)) if on]
    runs = {s: run_pieces(D, f"mtf_trendline_{s}@v1", "high_of_n@v1", "be_ladder@v1",
                          secondary=SECONDARY, params=PARAMS, cfg=COSTS,
                          entry_filter=_filter(s, veto)) for s in sides}
    t = pd.concat([trendline_trigger(r.df, r.trades, PARAMS, prefix_names=TF_NAMES)
                   for r in runs.values()], ignore_index=True) \
          .sort_values("entry_time").reset_index(drop=True)
    risk = (t.entry_price - t.stop_price).abs()
    t["R"] = t.pnl / (risk * t.qty)
    t["risk_%"] = 100 * risk / t.entry_price
    t["tid"] = [f"#{i+1:02d}" for i in range(len(t))]
    return t

ALL = _run(False)                      # every trade the trendline rules take
TRADES = _run(True) if USE_FVG_VETO else ALL
assert not len(TRADES) or TRADES.entry_time.min() >= _vs

if ENTRY_RULES:
    print(describe_rules(ENTRY_RULES, chart_tf=TF).to_string(index=False))
    print()
_g = {k: len(fvg_table(v, **FVG_KW)) for k, v in {TF: df, **GAP_FRAMES}.items()}
print(f"{len(df):,} {TF} bars | gaps found: " + "  ".join(f"{k} {v:,}" for k, v in _g.items()))
print(f"{len(TRADES)} trades   {TRADES.R.sum():+.1f}R   {100*(TRADES.pnl>0).mean():.0f}% win   "
      f"{100*TRADES.pnl.sum()/COSTS['initial_cash']:+.2f}%")

[20:11:49] INFO | quant.data | cache HIT  binance BTCUSDT 5m -> pushdown load
[20:11:49] INFO | quant.data | cache HIT  binance BTCUSDT 1h -> pushdown load
[20:11:49] INFO | quant.data | cache HIT  binance BTCUSDT 15m -> pushdown load
[20:11:49] INFO | quant.data | cache HIT  binance BTCUSDT 4h -> pushdown load
[20:11:49] INFO | quant.data | cache HIT  binance BTCUSDT 1h -> pushdown load
[20:11:49] INFO | quant.data | cache HIT  binance BTCUSDT 15m -> pushdown load
[20:11:50] INFO | quant.engine | backtest done | bars=11,521 trades=3 final_cash=9670.60 return=-3.29% dd=3.29% | 27.7 ms
[20:11:50] INFO | quant.engine | backtest done | bars=11,521 trades=5 final_cash=11015.55 return=10.16% dd=1.72% | 0.7 ms
[20:11:50] INFO | quant.engine | backtest done | bars=11,521 trades=2 final_cash=9781.63 return=-2.18% dd=2.18% | 0.5 ms
[20:11:50] INFO | quant.engine | backtest done | bars=11,521 trades=1 final_cash=10244.86 return=2.45% dd=0.92% | 0.4 ms


                                    rule  n   reference       column relation  side colour applies_to
long:  1 candles EMA_15m_100 close above  1 EMA_15m_100 15m__ema_100    close above    any       long
short: 1 candles EMA_15m_100 close below  1 EMA_15m_100 15m__ema_100    close below    any      short

11,521 5m bars | gaps found: 5m 1,128  4h 52  1h 55  15m 108
3 trades   +0.6R   33% win   +0.26%


## What the veto removed

The only question worth asking of a filter: *of the trades it blocked, how did they actually do?*

**It has already been asked properly, and the answer was no.**
[`experiments/fvg_veto_search.py`](../experiments/fvg_veto_search.py) ran 24 gate settings across
six windows spanning five months, scoring each by **R per trade** (the veto can only remove trades,
so total R falls even when the filter is doing nothing) with the top winner trimmed:

| | |
|---|---|
| settings clearing all 6 windows | **0** |
| expected by chance (24 × 0.5⁶) | 0.38 |
| settings winning ≥ 4 of 6 | 4 |
| expected by chance (24 × 34.4%) | **8.2** |

Four settings beat four windows where chance alone predicts eight. The veto is not merely
unproven — at every setting tried it performed **at or below** what random trade removal produces.

The table below describes *this* window only. On nine days the whole sim takes eight trades, and a
three-trade block that happens to be all losers arises by chance 7.1% of the time — so of the 96
gate combinations first tried here, about seven produced one for free. Read it as a picture of what
the filter did, never as a reason to keep it.

In [3]:
if USE_FVG_VETO and len(ALL):
    kept = set(zip(ALL.entry_time, ALL.side)) & set(zip(TRADES.entry_time, TRADES.side))
    blocked = ALL[~ALL.set_index(["entry_time", "side"]).index.isin(kept)]
    print(f"{len(ALL)} trades without the veto -> {len(TRADES)} with it "
          f"({len(blocked)} blocked)\n")
    rows = [("kept   ", TRADES), ("blocked", blocked), ("all    ", ALL)]
    print(f"{'':10s} {'n':>3s} {'R':>8s} {'R/trade':>9s} {'win%':>6s}")
    for nm, t in rows:
        if len(t):
            print(f"{nm:10s} {len(t):3d} {t.R.sum():+8.2f} {t.R.mean():+9.2f} "
                  f"{100*(t.pnl>0).mean():6.0f}")
        else:
            print(f"{nm:10s} {0:3d} {'-':>8s} {'-':>9s} {'-':>6s}")
    if len(blocked):
        print()
        print(blocked[["tid", "entry_time", "side", "cause", "R", "close_reason"]]
              .to_string(index=False))
else:
    print("veto is off — set USE_FVG_VETO = True in nb19_config.py")

8 trades without the veto -> 3 with it (5 blocked)

             n        R   R/trade   win%
kept         3    +0.61     +0.20     33
blocked      5    +6.39     +1.28     60
all          8    +6.99     +0.87     50

tid                entry_time  side  cause         R close_reason
#02 2026-06-02 07:00:00+00:00 short 15m·S5  2.871383  take_profit
#03 2026-06-05 01:00:00+00:00 short 5m·S49  2.886505  take_profit
#04 2026-06-05 18:50:00+00:00 short  1h·S7 -1.053178    stop_loss
#05 2026-06-05 22:50:00+00:00  long 15m·R8 -1.135041    stop_loss
#08 2026-06-09 09:30:00+00:00 short 15m·S7  2.818152  take_profit


## Chart

Gap boxes are drawn **behind** everything else, one legend entry per timeframe per direction —
click a legend row to show or hide that whole layer, or use the three dropdowns
(**timeframe → family → occurrence**) to isolate a single gap with its own hover.

Green is bullish (sits below price), red is bearish (sits above). A box runs from the bar the gap
became knowable to the bar that mitigated it; an unmitigated gap runs to the right edge.

In [4]:
from quant.viz import pattern_figure

fig = pattern_figure(
    D, patterns=[], trendlines=SHOW_TRENDLINES, trendline_kwargs=TRENDLINE,
    tf_frames=LINE_FRAMES, fvg_frames=GAP_FRAMES,
    fvgs=FVG_DRAW, fvg_kwargs=FVG_KW, fvg_extend=FVG_EXTEND, fvg_show=tuple(FVG_SHOW),
    max_fvg_picker=MAX_FVG_PICKER,
    view_start=VIEW_START, emas=EMAS, mtf_emas=MTF_EMAS, emas_on=EMAS_ON, chart_tf=TF,
    ema_frames={**LINE_FRAMES, **GAP_FRAMES},
    sr_zones=SHOW_SR_ZONES, zone_lookback_bars=ZONE_LOOKBACK,
    max_picker_instances=MAX_PICKER, max_shapes_per_tf=MAX_SHAPES_TF,
    max_trade_picker=MAX_TRADE_PICKER,
    trades=TRADES, trade_tp_rr=TP_RR,
    start_visible=("trades", "fvg_bull", "fvg_bear"),
    title=f"{SYMBOL} {TF}  {VIEW_START} to {END}  ·  "
          f"{'/'.join((TF,) + tuple(FVG_TFS))} fair value gaps  ·  {len(TRADES)} trades",
    height=HEIGHT)
# fig

Open it in a browser tab — a full-width chart beats a notebook cell for this.

In [5]:
from quant.viz import open_in_browser
open_in_browser(fig)          # or fig.write_html("fvg.html", include_plotlyjs="cdn")

WindowsPath('C:/Users/Talal Zahid/Desktop/Talal/Binance/notebooks/chart.html')

## Appendix — the gap table

One row per gap, with all three mitigation thresholds recorded whatever `FVG_MITIGATION` is set to.
`dead_i` is the one currently in force; `-1` means still open.

In [6]:
GAPS = {TF: fvg_table(df, **FVG_KW)}
for _k, _f in GAP_FRAMES.items():
    GAPS[_k] = fvg_table(_f.reset_index(drop=True), **FVG_KW)

for _k, _t in GAPS.items():
    if not len(_t):
        print(f"{_k}: no gaps"); continue
    _open = _t[_t.dead_i < 0]
    print(f"{_k}: {len(_t):,} gaps, {len(_open)} still open, "
          f"median size {_t.size_atr.median():.2f} ATR, "
          f"median bars to mitigate "
          f"{(_t.dead_i - _t.born_i)[_t.dead_i >= 0].median():.0f}")

_show = GAPS[TF]
_show[_show.dead_i < 0].tail(12)[["id", "side", "born_t", "bot", "top", "size_atr"]]

5m: 1,128 gaps, 62 still open, median size 0.78 ATR, median bars to mitigate 5
4h: 52 gaps, 8 still open, median size 0.77 ATR, median bars to mitigate 6
1h: 55 gaps, 6 still open, median size 0.82 ATR, median bars to mitigate 4
15m: 108 gaps, 7 still open, median size 0.80 ATR, median bars to mitigate 6


,id,side,born_t,bot,top,size_atr
982,FS501,bearish,2026-06-02 04:50:00+00:00,70916.00,70994.47,0.722232
1005,FS513,bearish,2026-06-03 11:35:00+00:00,67192.00,67308.00,1.043317
1008,FS515,bearish,2026-06-03 15:10:00+00:00,66669.86,66760.00,0.508425
1015,FS521,bearish,2026-06-03 22:15:00+00:00,65328.00,65460.99,0.595702
1043,FS535,bearish,2026-06-05 05:40:00+00:00,63216.00,63335.03,0.520919
1072,FB525,bullish,2026-06-07 01:05:00+00:00,60811.99,60982.00,1.324639
1089,FB536,bullish,2026-06-07 20:05:00+00:00,61358.00,61486.00,0.815049
1111,FS562,bearish,2026-06-08 22:05:00+00:00,63581.12,63728.96,1.368409
1118,FS567,bearish,2026-06-09 07:30:00+00:00,63179.98,63300.00,1.054415
1119,FS568,bearish,2026-06-09 08:05:00+00:00,63076.41,63182.95,0.956170
